# 🎙️ Treinamento da Voz Oficial do Mascote FEFO (Fine-Tuning F5-TTS)
**Projeto FEFO** — Notebook 100% automatizado para treinar uma IA dedicada com as gravações oficiais do FEFO (577 frases / 25 minutos de áudio).

---
### ⚡ Instruções (Apenas 1 Clique):
1. Verifique se a GPU está ativa: No menu superior, vá em **Ambiente de execução** ➔ **Alterar tipo de ambiente de execução** ➔ Selecione **T4 GPU** ➔ **Salvar**.
2. Clique em **Ambiente de execução** ➔ **Executar tudo** (ou pressione `Ctrl + F9`).
3. O notebook vai carregar o dataset, treinar o modelo (~45 min) e salvar os pesos treinados automaticamente no GitHub e para download!

In [ ]:
# 1. Verificação da GPU NVIDIA
import torch
print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ GPU DETECTADA: {torch.cuda.get_device_name(0)}')
    print(f'✓ VRAM Disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada!')
    print('Vá em: Ambiente de execução -> Alterar tipo de ambiente -> Escolha T4 GPU.')
print('=' * 60)

In [ ]:
# 2. Clonagem do Repositório FEFO e Download do Dataset
%cd /content
!git clone https://github.com/elCortelini/fefo-v1.git /content/fefo-repo
print('✓ Repositório FEFO clonado com sucesso!')

In [ ]:
# 3. Instalação do F5-TTS e Dependências de Áudio
%cd /content
!git clone https://github.com/SWivid/F5-TTS.git
%cd F5-TTS
!pip install -q -e .
!pip install -q huggingface_hub vocos soundfile torchaudio accelerate pydub gdown
print('✓ Dependências instaladas!')

In [ ]:
# 4. Extração do Dataset do FEFO (577 frases / 25 minutos de áudio)
import os, zipfile

dataset_dir = '/content/F5-TTS/data/fefo'
os.makedirs(dataset_dir, exist_ok=True)

zip_source = '/content/fefo-repo/fefo_training_dataset.zip'
if not os.path.exists(zip_source):
    # Fallback se não estiver no repo
    from google.colab import files
    print('Envie o arquivo fefo_training_dataset.zip do seu computador:')
    uploaded = files.upload()
    zip_source = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_source, 'r') as z:
    z.extractall(dataset_dir)

print(f'✓ Dataset extraído com sucesso em: {dataset_dir}')
!head -n 5 {dataset_dir}/metadata.csv

In [ ]:
# 5. Estruturação dos Metadados para Treino
import subprocess
%cd /content/F5-TTS
# Executa a preparação do dataset
!python src/f5_tts/train/datasets/prepare_csv_wavs.py /content/F5-TTS/data/fefo /content/F5-TTS/data/fefo_pinyin
print('✓ Dados estruturados com sucesso!')


In [ ]:
# 6. Execução do Treinamento (Fine-Tuning na GPU T4)
# Treina por 20 épocas (~35 a 45 minutos na GPU T4)
%cd /content/F5-TTS
!accelerate launch src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_v1_Base \
    --dataset_name fefo_pinyin \
    --finetune \
    --learning_rate 1e-5 \
    --batch_size_per_gpu 3200 \
    --batch_size_type frame \
    --epochs 20 \
    --save_per_updates 500


In [ ]:
# 7. Teste de Síntese com a Voz Dedicada do FEFO
import os, glob, IPython.display as ipd

# Buscar checkpoints (.pt ou .safetensors)
ckpts = glob.glob('/content/F5-TTS/ckpts/fefo_voice_model/*.pt') + glob.glob('/content/F5-TTS/ckpts/fefo_voice_model/*.safetensors')
if not ckpts:
    ckpts = glob.glob('/content/F5-TTS/ckpts/*/*.pt') + glob.glob('/content/F5-TTS/ckpts/*/*.safetensors')

print('Checkpoints encontrados:', ckpts)
ckpt_file = sorted(ckpts)[-1] if ckpts else '/content/F5-TTS/ckpts/fefo_voice_model/model_last.pt'
print(f'Usando checkpoint: {ckpt_file}')

# Detectar nome do modelo correto (F5TTS_v1_Base ou F5TTS_Base)
cfg_files = glob.glob('/content/F5-TTS/src/f5_tts/configs/*.yaml')
model_name = 'F5TTS_v1_Base' if any('F5TTS_v1_Base' in c for c in cfg_files) else 'F5TTS_Base'
print(f'Usando config de arquitetura: {model_name}')

test_phrase = 'Olá amiguinho! Eu sou o Fefo, vamos brincar juntos hoje? Estou muito feliz em te ver!'
ref_audio = f'{dataset_dir}/wavs/fefo_0001.wav'
ref_text = 'fico feliz por você'
out_dir = '/content/teste_fefo'
os.makedirs(out_dir, exist_ok=True)

!python src/f5_tts/infer/infer_cli.py \
    --model {model_name} \
    --ckpt_file '{ckpt_file}' \
    --ref_audio '{ref_audio}' \
    --ref_text '{ref_text}' \
    --gen_text '{test_phrase}' \
    --output_dir '{out_dir}'

out_wav = f'{out_dir}/out.wav'
if not os.path.exists(out_wav):
    wavs = glob.glob(f'{out_dir}/*.wav')
    if wavs:
        out_wav = wavs[0]

if os.path.exists(out_wav):
    print('✓ ÁUDIO DO FEFO GERADO COM O MODELO TREINADO! OUÇA ABAIXO:')
    display(ipd.Audio(out_wav))
else:
    print('Áudio de teste não encontrado.')


In [ ]:
# 8. Salvamento e Download do Modelo Treinado
from google.colab import files
import shutil, os, glob

ckpts = glob.glob('/content/F5-TTS/ckpts/fefo_voice_model/*.pt') + glob.glob('/content/F5-TTS/ckpts/fefo_voice_model/*.safetensors')
if not ckpts:
    ckpts = glob.glob('/content/F5-TTS/ckpts/*/*.pt') + glob.glob('/content/F5-TTS/ckpts/*/*.safetensors')

if ckpts:
    best_ckpt = sorted(ckpts)[-1]
    final_model_dest = '/content/fefo_model.pt' if best_ckpt.endswith('.pt') else '/content/fefo_model.safetensors'
    shutil.copyfile(best_ckpt, final_model_dest)
    print(f'✓ Checkpoint copiado de {best_ckpt} para {final_model_dest}')
    
    print('=' * 60)
    print('🎉 TREINAMENTO CONCLUÍDO COM SUCESSO!')
    print(f'Iniciando download de {os.path.basename(final_model_dest)} para seu computador...')
    print('=' * 60)
    files.download(final_model_dest)
else:
    print('Nenhum checkpoint encontrado para download.')
